In [2]:
from pathlib import Path
import numpy as np
import pandas as pd


def count_neurons_in_npy(npy_path: Path) -> int:
    """
    Load a .npy file containing kept neuron indices and return the number of neurons.
    """
    arr = np.load(npy_path, allow_pickle=True)

    # Usually this file contains a 1D array of neuron indices.
    # Using .size is robust for numpy arrays.
    if isinstance(arr, np.ndarray):
        return int(arr.size)

    # Fallback in case the loaded object is not a standard ndarray
    try:
        return len(arr)
    except TypeError:
        return 1


def find_kept_neuron_file(session_folder: Path) -> Path | None:
    """
    Build the expected file path for one session folder.

    Example:
    session folder: L472_f03_Exp_2_rocking_1
    expected file:  L472_f03_kept_neuron_indices.npy
    """
    parts = session_folder.name.split("_")

    # We expect at least something like: L472_f03_...
    if len(parts) < 2:
        return None

    prefix = f"{parts[0]}_{parts[1]}"
    target_file = (
        session_folder
        / "03_analysis"
        / "functional"
        / "plots"
        / "filtered_neurons_by_stimuli"
        / f"{prefix}_kept_neuron_indices.npy"
    )

    if target_file.exists():
        return target_file

    return None


def main():
    base_path = Path(r"D:\Alejandro\Data\OneDrive - Université de Lausanne\Lab\Data\2p")

    results = []

    # Only direct subfolders that start with "L"
    for folder in sorted(base_path.iterdir()):
        if not folder.is_dir():
            continue
        if not folder.name.startswith("L"):
            continue

        npy_file = find_kept_neuron_file(folder)
        if npy_file is None:
            print(f"Skipping (file not found): {folder.name}")
            continue

        try:
            neuron_count = count_neurons_in_npy(npy_file)

            results.append({
                "folder_name": folder.name,
                "file_name": npy_file.name,
                "neuron_count": neuron_count,
            })

            print(f"OK: {folder.name} -> {neuron_count} neurons")

        except Exception as e:
            print(f"Error reading {npy_file}: {e}")
            continue

    if not results:
        print("No valid kept_neuron_indices.npy files were found.")
        return

    df = pd.DataFrame(results)
    df = df.sort_values("folder_name").reset_index(drop=True)

    total_neurons = int(df["neuron_count"].sum())

    # Add a total row at the bottom
    total_row = pd.DataFrame([{
        "folder_name": "TOTAL",
        "file_name": "",
        "neuron_count": total_neurons,
    }])

    df_out = pd.concat([df, total_row], ignore_index=True)

    output_file = base_path / "kept_neuron_counts_summary.xlsx"
    df_out.to_excel(output_file, index=False)

    print("\nDone.")
    print(f"Excel file saved to: {output_file}")
    print(f"Total neurons: {total_neurons}")


if __name__ == "__main__":
    main()

OK: L433_f02_Exp_1_flickering -> 641 neurons
OK: L433_f03_Exp_1_flickering -> 268 neurons
OK: L433_f04_Exp_1_flickering -> 551 neurons
OK: L433_f05_Exp_1_flickering -> 432 neurons
OK: L433_f06_Exp_1_flickering -> 438 neurons
OK: L453_f07_Exp_1_flickering -> 72 neurons
OK: L453_f08_Exp_1_flickering -> 534 neurons
OK: L453_f09_Exp_1_flickering -> 530 neurons
OK: L453_f10_Exp_1_flickering -> 219 neurons
OK: L453_f11_Exp_1_flickering -> 426 neurons
OK: L472_f01_Exp_2_rocking_1 -> 54 neurons
OK: L472_f02_Exp_2_rocking_1 -> 378 neurons
OK: L472_f03_Exp_2_rocking_1 -> 325 neurons
OK: L472_f04_Exp_2_rocking_1 -> 142 neurons
OK: L472_f05_Exp_2_rocking_1 -> 19 neurons
OK: L472_f06_Exp_2_rocking_1 -> 422 neurons
OK: L588_f01_Exp_3_rocking_2 -> 315 neurons
OK: L588_f02_Exp_3_rocking_2 -> 340 neurons
OK: L588_f03_Exp_3_rocking_2 -> 361 neurons
OK: L588_f04_Exp_3_rocking_2 -> 78 neurons
OK: L588_f05_Exp_3_rocking_2 -> 284 neurons
OK: L588_f06_Exp_3_rocking_2 -> 105 neurons
OK: L588_f07_Exp_3_rocking